# test_photo_9 폴더 예측
Drive에 test_photo_9 폴더와 본인이 학습한 EfficientNet-B0 best.pt를 올린 뒤 경로를 지정하세요. 재학습 없이 실행합니다. HTML은 이미지가 내장되어 단독으로 열립니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
"""Predict a photo folder using this project's trained EfficientNet-B0 checkpoint."""
import argparse
import base64
import csv
import html
import io
import json
import math
from pathlib import Path
from PIL import Image, ImageOps



def save_png_grid(rows, output):
    """Render nine images per PNG, with prediction captions."""
    from PIL import ImageDraw, ImageFont
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    fonts = ["C:/Windows/Fonts/malgun.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]
    font_path = next((p for p in fonts if Path(p).is_file()), None)
    font = ImageFont.truetype(font_path, 26) if font_path else ImageFont.load_default(size=26)
    small = ImageFont.truetype(font_path, 20) if font_path else ImageFont.load_default(size=20)
    targets = []
    for start in range(0, len(rows), 9):
        canvas = Image.new("RGB", (1260, 1530), "#f5f6f8")
        draw = ImageDraw.Draw(canvas)
        for index, row in enumerate(rows[start:start + 9]):
            x, y = 20 + (index % 3) * 410, 20 + (index // 3) * 500
            draw.rounded_rectangle((x, y, x + 400, y + 480), radius=12, fill="white")
            with Image.open(row["path"]) as image:
                image = ImageOps.exif_transpose(image).convert("RGB")
                image.thumbnail((376, 376), Image.Resampling.LANCZOS)
                canvas.paste(image, (x + (400-image.width)//2, y + 12 + (376-image.height)//2))
            caption = f"Prediction: {float(row['score']):.4f} / 5"
            draw.text((x+200, y+408), caption, font=font, fill="#172033", anchor="mm")
            name = row["filename"]
            while draw.textlength(name, font=small) > 376 and len(name) > 4:
                name = name[:-4] + "..."
            draw.text((x+200, y+449), name, font=small, fill="#555555", anchor="mm")
        suffix = "" if len(rows) <= 9 else f"_{start//9+1:02d}"
        target = output / f"test_photo_9_predictions{suffix}.png"
        canvas.save(target, dpi=(150,150))
        targets.append(target)
    return targets

def save_report(rows, config, checkpoint_path, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    cards = []
    for row in rows:
        with Image.open(row['path']) as image:
            image = ImageOps.exif_transpose(image).convert('RGB')
            image.thumbnail((360, 360))
            stream = io.BytesIO()
            image.save(stream, format='JPEG', quality=85)
        data = base64.b64encode(stream.getvalue()).decode('ascii')
        name = html.escape(row['filename'])
        cards.append(f'<article><img alt="{name}" src="data:image/jpeg;base64,{data}">'
                     f'<div>AI 모델 예측 점수</div><strong>{row["score"]:.4f} / 5</strong>'
                     f'<p>{name}</p></article>')
    metadata = dict(config, checkpoint=str(checkpoint_path), image_count=len(rows))
    page = '''<!doctype html><html lang="ko"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>얼굴 이미지 예측 점수</title><style>
body{font-family:Arial,sans-serif;margin:24px;background:#f5f6f8;color:#172033}
.grid{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:16px}
article{min-width:0;text-align:center;background:white;padding:12px;border-radius:12px}
img{width:100%;height:240px;object-fit:contain}strong{display:block;font-size:22px;margin-top:6px}
p{overflow-wrap:anywhere}pre{white-space:pre-wrap;overflow-wrap:anywhere}
</style></head><body><h1>얼굴 이미지 예측 점수</h1>
<p>점수 범위 1~5 · 정답 라벨이 없는 별도 입력 이미지의 예측 결과</p>'''
    page += '<details><summary>모델 및 학습 설정</summary><pre>'
    page += html.escape(json.dumps(metadata, ensure_ascii=False, indent=2)) + '</pre></details>'
    page += '<div class="grid">' + ''.join(cards) + '</div></body></html>'
    target = output / 'test_photo_9_predictions.html'
    target.write_text(page, encoding='utf-8')
    with (output / 'test_photo_9_predictions.csv').open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['filename', 'path', 'score'])
        writer.writeheader()
        writer.writerows(rows)
    (output / 'prediction_config.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    for png in save_png_grid(rows, output):
        print('PNG:', png)
    return target


def predict_folder(checkpoint_path, images, output):
    import torch
    from torch import nn
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

    checkpoint_path, images = Path(checkpoint_path), Path(images)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f'EfficientNet-B0 best.pt 경로를 지정하세요: {checkpoint_path}')
    if not images.is_dir():
        raise FileNotFoundError(f'이미지 폴더 없음: {images}')
    files = sorted(p for p in images.iterdir() if p.is_file() and p.suffix.lower()
                   in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'})
    if not files:
        raise ValueError('이미지 폴더가 비어 있습니다.')
    # Use only a checkpoint you created/trust: legacy config values require pickle loading.
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    config = checkpoint['config']
    if config.get('model') != 'efficientnet_b0':
        raise ValueError('이 코드는 EfficientNet-B0 체크포인트 전용입니다.')
    task = config['task']
    if task not in {'regression', 'classification'}:
        raise ValueError(f'지원하지 않는 task: {task}')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 5 if task == 'classification' else 1)
    model.load_state_dict(checkpoint['state_dict'], strict=True)
    model.to(device).eval()
    transform = EfficientNet_B0_Weights.IMAGENET1K_V1.transforms()
    rows = []
    with torch.inference_mode():
        for path in files:
            with Image.open(path) as image:
                tensor = transform(ImageOps.exif_transpose(image).convert('RGB')).unsqueeze(0).to(device)
            logits = model(tensor)
            score = ((logits.softmax(1) * torch.arange(1, 6, device=device)).sum(1).item()
                     if task == 'classification' else (1 + 4 * logits.squeeze(1).sigmoid()).item())
            if not math.isfinite(score) or not 1 <= score <= 5:
                raise ValueError(f'비정상 예측: {path.name}: {score}')
            rows.append({'filename': path.name, 'path': str(path.resolve()), 'score': score})
            print(f'{path.name}: {score:.4f}')
    target = save_report(rows, config, checkpoint_path, output)
    print(f'{len(rows)}개 이미지 완료: {target}')
    return target




In [ ]:
PROJECT = Path("/content/drive/MyDrive/image_attractive_visionProject")
CHECKPOINT = PROJECT / "best.pt"  # 실제 저장 위치로 수정
IMAGES = PROJECT / "test_photo_9"
OUTPUT = PROJECT / "colab_outputs/test_photo_9"
report_path = predict_folder(CHECKPOINT, IMAGES, OUTPUT)
from IPython.display import HTML, display
from IPython.display import Image as DisplayImage
for png_path in sorted(OUTPUT.glob("test_photo_9_predictions*.png")):
    display(DisplayImage(filename=str(png_path)))
local_output = Path("/content/output")
local_output.mkdir(parents=True, exist_ok=True)
import shutil
for file in OUTPUT.iterdir():
    if file.is_file():
        shutil.copy2(file, local_output / file.name)
print("Colab 결과:", local_output)
